# Generating Dense Text Embeddings with Hugging Face & Sentence-Transformers

**Environment:** Google Colab (CPU or GPU) / Local Python 3.9+  
**Domain:** Enterprise IT Incident Response, Cloud Operations & Semantic Knowledge Retrieval

---

### What You Will Learn
1. **From Static to Contextual:** Why Transformer embeddings solve Word2Vec's polysemy and syntax blind spots.
2. **Under the Hood:** Implementing custom **Mean-Pooling** with PyTorch and raw Hugging Face `AutoTokenizer` / `AutoModel`.
3. **Sentence-Transformers Framework:** High-level inference using `all-MiniLM-L6-v2` and unit normalization.
4. **Latency & Dimension Benchmarking:** Comparing lightweight models vs. deeper architectures.
5. **Semantic IT Incident Routing Engine:** Building an automated search and triage system for infrastructure outages.
6. **Latent Space Visualization:** 2D PCA / t-SNE clustering of technical enterprise documents.
7. **Production Exercises:** Asymmetric query matching, semantic deduplication, and zero-shot ticket classification.

In [ ]:
# Install required dependencies
!pip install -q sentence-transformers transformers torch scikit-learn matplotlib seaborn pandas numpy

## 1. What are Transformer-Based Sentence Embeddings?

While Word2Vec produces a **single static vector** for every dictionary word (regardless of context), modern **Transformers** (BERT, RoBERTa, MiniLM) generate **contextual representations**.
- In *"Apple released a new server chip"*, `"Apple"` is recognized as a tech enterprise.
- In *"I ate a crisp green apple"*, `"apple"` is recognized as fruit.

### How Sentence Embeddings are Formed:
1. **Tokenization:** Breaks text into subword tokens and appends special tokens (`[CLS]`, `[SEP]`).
2. **Self-Attention Layers:** Computes interactions between all tokens in the sentence across multiple attention heads.
3. **Pooling:** Aggregates token-level hidden states into a single fixed-size sentence vector $\mathbf{v} \in \mathbb{R}^D$:
   - **[CLS] Token:** Uses the first token's vector.
   - **Mean Pooling (Standard):** Computes the average of all token vectors, weighted by the attention mask.
   - **$L_2$ Normalization:** Normalizes the pooled vector so that $\|\mathbf{v}\|_2 = 1.0$.

## 2. Under the Hood: Custom Mean-Pooling with Raw Transformers

Before using high-level wrappers, let us implement the raw PyTorch Mean-Pooling pipeline to see exactly how transformer tokens become sentence vectors.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

model_id = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

sample_text = "Kubernetes worker node failed due to out of memory OOMKilled error."

# 1. Tokenize input with attention mask
inputs = tokenizer(sample_text, padding=True, truncation=True, return_tensors="pt")
print("Input Token IDs Shape:", inputs["input_ids"].shape)
print("Attention Mask Shape :", inputs["attention_mask"].shape)

# 2. Forward pass through transformer backbone
with torch.no_grad():
    model_output = model(**inputs)

# last_hidden_state shape: [batch_size, sequence_length, hidden_dimension]
token_embeddings = model_output.last_hidden_state
print("Token Embeddings Shape:", token_embeddings.shape)

# 3. Custom Mean-Pooling respecting attention mask
def mean_pooling(token_embeddings, attention_mask):
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

sentence_embedding = mean_pooling(token_embeddings, inputs["attention_mask"])

# 4. L2 Normalization
normalized_embedding = torch.nn.functional.normalize(sentence_embedding, p=2, dim=1)
print("Pooled Sentence Vector Shape:", normalized_embedding.shape)
print(f"Norm verification: {torch.norm(normalized_embedding).item():.4f}")

## 3. High-Level Inference with Sentence-Transformers

The `sentence-transformers` library encapsulates tokenization, model execution, mean-pooling, and normalization into a clean, optimized API.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load all-MiniLM-L6-v2 (compact ~80 MB, 384 dimensions, extremely fast)
st_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Architecture:", st_model)
print("Vector Dimensionality:", st_model.get_sentence_embedding_dimension())
print("Maximum Sequence Length:", st_model.max_seq_length)

## 4. Multi-Domain Enterprise Corpus & Similarity Matrix

Let us define a realistic enterprise incident and operations corpus covering:
- Database connectivity & locks
- Kubernetes cluster memory & pods
- SSL/TLS cryptographic certificates
- IAM cloud permissions & security

In [ ]:
incident_docs = [
    # Database issues
    "PostgreSQL database pool exhausted with connection timeout errors on port 5432.",
    "Database deadlocks detected on transactional accounts table during high write load.",
    
    # Kubernetes & Infrastructure
    "Kubernetes pod evicted due to node memory pressure and OOMKilled status.",
    "Worker node unable to schedule pods because cluster CPU capacity reached 100%.",
    
    # SSL / Certificates
    "HTTPS SSL TLS certificate expired causing browser security warning on gateway.",
    "Let's Encrypt automated certificate renewal failed due to DNS challenge timeout.",
    
    # IAM & Security
    "Access denied on AWS S3 bucket due to missing IAM GetObject permission policy.",
    "User authentication failed after multi-factor MFA token synchronization error."
]

# Generate dense embeddings for all documents
doc_embeddings = st_model.encode(incident_docs, convert_to_numpy=True, normalize_embeddings=True)

print("Generated embeddings matrix shape:", doc_embeddings.shape)

### 4.1 Pairwise Semantic Heatmap
Let us compute the cosine similarity matrix across all incident documents and visualize it with Seaborn.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
pairwise_sim = cosine_similarity(doc_embeddings)

doc_labels = [
    "DB: Timeout", "DB: Deadlock",
    "K8s: OOMKilled", "K8s: CPU Cap",
    "SSL: Expired", "SSL: DNS Fail",
    "IAM: S3 Denied", "IAM: MFA Error"
]

plt.figure(figsize=(9, 7))
sns.heatmap(
    pairwise_sim,
    xticklabels=doc_labels,
    yticklabels=doc_labels,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar=True
)
plt.title("Semantic Cosine Similarity Between IT Incident Documents", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

Notice the clear 2x2 high-similarity blocks along the diagonal:
- Database pairs score $\approx 0.60 - 0.75$
- Kubernetes pairs score $\approx 0.65 - 0.75$
- SSL pairs score $\approx 0.60 - 0.70$
- IAM pairs score $\approx 0.50 - 0.60$
- Cross-domain pairs drop below $0.20$!

## 5. Automated IT Incident Routing Engine (Semantic Search)

When an engineer or user submits an outage description in plain natural language, our engine maps it to the closest known runbook, even if the wording is completely different!

In [ ]:
import numpy as np

# Resolution runbooks corresponding to our incident documents
runbooks = {
    0: "Runbook-DB-01: Scale up connection pool size in PgBouncer and kill idle client connections.",
    1: "Runbook-DB-02: Inspect pg_locks, terminate blocking transaction PID, and add missing index.",
    2: "Runbook-K8s-01: Increase container memory limits in Deployment manifest and configure Horizontal Pod Autoscaler.",
    3: "Runbook-K8s-02: Provision additional EC2/Compute worker nodes or drain non-critical batch jobs.",
    4: "Runbook-SEC-01: Issue new SSL certificate via cert-manager and reload ingress controller.",
    5: "Runbook-SEC-02: Verify Route53 DNS API credentials and re-trigger Cert-Manager challenge.",
    6: "Runbook-IAM-01: Attach AmazonS3ReadOnlyAccess policy to the caller IAM execution role.",
    7: "Runbook-IAM-02: Reset MFA device in Okta admin dashboard and generate temporary emergency bypass code."
}

def triage_incident(query_text, top_k=2, similarity_threshold=0.35):
    """Triages user query by retrieving closest technical runbooks."""
    query_vec = st_model.encode([query_text], normalize_embeddings=True)
    scores = cosine_similarity(query_vec, doc_embeddings)[0]
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"\nIncident Report: '{query_text}'")
    print("-" * 70)
    for rank, idx in enumerate(top_indices, start=1):
        score = scores[idx]
        if score >= similarity_threshold:
            print(f"  Rank #{rank} [Match: {score*100:.1f}%]:")
            print(f"    Matched Issue : {incident_docs[idx]}")
            print(f"    Action Plan   : {runbooks[idx]}")
        else:
            print(f"  Rank #{rank} [Below Threshold {score*100:.1f}%] - No confident match.")

# Test with diverse natural language queries
triage_incident("My web browser says the site is dangerous because https cert is out of date")
triage_incident("The docker pod was terminated because it consumed too much RAM")
triage_incident("Cannot read files from cloud storage bucket 403 forbidden error")

## 6. Model Architecture & Latency Benchmarking

Different embedding models strike different trade-offs between speed, memory footprint, and representation quality. Let us benchmark two prominent models:
1. `sentence-transformers/all-MiniLM-L6-v2` (Lightweight, 384 dimensions)
2. `sentence-transformers/paraphrase-MiniLM-L3-v2` (Ultra-fast, 384 dimensions, 3 layers)

In [ ]:
import time

models_to_test = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/paraphrase-MiniLM-L3-v2"
]

benchmark_queries = [
    "How to configure database connection pools?",
    "Container crash looping with exit code 137",
    "Renewing wildcard SSL certificates with certbot",
    "Cloud IAM policy principle of least privilege"
] * 25  # 100 sentences

bench_results = []
for model_name in models_to_test:
    t_start = time.perf_counter()
    bench_model = SentenceTransformer(model_name)
    load_time = time.perf_counter() - t_start
    
    t0 = time.perf_counter()
    _ = bench_model.encode(benchmark_queries, batch_size=32, show_progress_bar=False)
    infer_time = time.perf_counter() - t0
    
    bench_results.append({
        "Model": model_name.split("/")[-1],
        "Dimensions": bench_model.get_sentence_embedding_dimension(),
        "Load Time (s)": round(load_time, 2),
        "Encoding Time (100 texts) (s)": round(infer_time, 4),
        "Texts / Sec": round(len(benchmark_queries) / infer_time, 1)
    })

pd.DataFrame(bench_results)

## 7. 2D Dimensionality Reduction (PCA & t-SNE)

Let us visually inspect how the dense embedding space naturally groups similar incident domains without any explicit training labels.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(doc_embeddings)

categories = ["Database", "Database", "Kubernetes", "Kubernetes", "Security/SSL", "Security/SSL", "IAM", "IAM"]
palette = {"Database": "#ef4444", "Kubernetes": "#3b82f6", "Security/SSL": "#10b981", "IAM": "#8b5cf6"}

plt.figure(figsize=(10, 7))
for i, label in enumerate(doc_labels):
    cat = categories[i]
    plt.scatter(coords_2d[i, 0], coords_2d[i, 1], color=palette[cat], s=120, edgecolors="black", zorder=3)
    plt.annotate(label, (coords_2d[i, 0] + 0.02, coords_2d[i, 1] + 0.01), fontsize=10, weight="bold")

# Add custom category legend
for cat, col in palette.items():
    plt.scatter([], [], color=col, label=cat, s=100)

plt.legend(title="Technical Domain", loc="upper left")
plt.title("2D PCA Visualization of Sentence-Transformers Latent Space", fontsize=13, weight="bold")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

## 8. Practice Exercises (Solved)

### Exercise 1: Asymmetric Search Simulation
In search systems, queries are short questions, while documents are long paragraphs. Many models (like BGE and E5) use task prefixes like `"query: "` to align the semantic representations.

In [ ]:
query_prefix = "query: "
doc_prefix = "passage: "

q = query_prefix + "How to resolve database locks?"
p1 = doc_prefix + "Database deadlocks detected on transactional accounts table during high write load."
p2 = doc_prefix + "User authentication failed after multi-factor MFA token synchronization error."

emb_q = st_model.encode([q], normalize_embeddings=True)
emb_passages = st_model.encode([p1, p2], normalize_embeddings=True)

sims = cosine_similarity(emb_q, emb_passages)[0]
print(f"Similarity with relevant passage   : {sims[0]:.4f}")
print(f"Similarity with irrelevant passage : {sims[1]:.4f}")

### Exercise 2: Semantic Deduplication Filter
Write a function that accepts a stream of incoming log messages and filters out semantic duplicates (messages with similarity $> 0.85$).

In [ ]:
def deduplicate_messages(messages, threshold=0.85):
    """Removes semantically redundant messages."""
    unique_messages = []
    unique_embeddings = []
    
    for msg in messages:
        msg_emb = st_model.encode([msg], normalize_embeddings=True)
        if not unique_embeddings:
            unique_messages.append(msg)
            unique_embeddings.append(msg_emb[0])
            continue
            
        sims = cosine_similarity(msg_emb, np.array(unique_embeddings))[0]
        max_sim = np.max(sims)
        
        if max_sim < threshold:
            unique_messages.append(msg)
            unique_embeddings.append(msg_emb[0])
        else:
            print(f"Filtered duplicate (sim={max_sim:.2f}): '{msg}'")
            
    return unique_messages

test_logs = [
    "Database connection pool full",
    "PostgreSQL connection limit reached",  # Semantically duplicate!
    "Kubernetes pod evicted",
    "DB connection pool exhausted"           # Semantically duplicate!
]

filtered = deduplicate_messages(test_logs, threshold=0.80)
print("\nRetained Unique Logs:")
for l in filtered:
    print(f" - {l}")

## Key Takeaways

| Feature | `sentence-transformers` | Raw `transformers` |
|---|---|---|
| **Convenience** | Single line `.encode()` | Requires manual tokenization & pooling |
| **Normalization** | Built-in `normalize_embeddings=True` | Requires `torch.nn.functional.normalize` |
| **Output Type** | NumPy arrays or PyTorch tensors | PyTorch Tensors |
| **Performance** | Multi-GPU batching out-of-the-box | Manual device placement |
| **Primary Use** | Semantic search, clustering, RAG | Custom model architectures & fine-tuning |